# Поиск аномалий

Методы обнаружения аномалий, как следует из названия, позволяют находить необычные объекты в выборке. Но что такое "необычные" и совпадает ли это определение у разных методов?

Начнём с поиска аномалий в текстах: научимся отличать вопросы о программировании от текстов из 20newsgroups про религию.

Подготовьте данные: в обучающую выборку возьмите 20 тысяч текстов из датасета Stack Overflow, а тестовую выборку сформируйте из 10 тысяч текстов со Stack Overflow и 100 текстов из класса soc.religion.christian датасета 20newsgroups (очень пригодится функция `fetch_20newsgroups(categories=['soc.religion.christian'])`). Тексты про программирование будем считать обычными, а тексты про религию — аномальными.

In [1]:
import sklearn as sk
from sklearn.model_selection import train_test_split

In [2]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(
    categories=['soc.religion.christian'],
    shuffle=True,
    random_state=42,

)

texts = data.data[:100]


In [3]:
import pandas as pd
import numpy as np

df = pd.read_parquet("post_questions_test_000000000000.parquet")
df = df["body"].reset_index(drop = True)
x_train, x_test = train_test_split(df, train_size = 20000, test_size = 10000, random_state = 52, shuffle = True)
y_train = pd.Series([0] * len(x_train))
y_test = pd.Series([0] * len(x_test) + [1] * len(texts))
texts = pd.Series(texts)
x_test = pd.concat([x_test, texts])
x_train = x_train.reset_index(drop=True)
x_test = x_test.reset_index(drop = True)

perm = np.random.permutation(len(x_test))
x_test = x_test.iloc[perm]
y_test = y_test.iloc[perm]
x_train = x_train.reset_index(drop=True)
x_test = x_test.reset_index(drop = True)
print(x_test[10000:10100])

10000    <pre><code> //this is my code that i have writ...
10001    <p>In the code snippet below I am trying to im...
10002    <p>I have a 'time remaining' counter in place ...
10003    <p>I am getting an error like <strong>Componen...
10004    <p>I am building an component which accept an ...
                               ...                        
10095    <p>I have a Dockerfile that is supposed to bui...
10096    <p>I am trying to use this code to resize my t...
10097    <p>Cannot convert lambda expression to type 'b...
10098    <p>I have <code>A.js</code>, <code>B.js</code>...
10099    <p>I am trying to plot figures in real time us...
Length: 100, dtype: object


**(1 балл)**

Проверьте качество выделения аномалий (pre и rec на тестовой выборке, если считать аномалии положительным классов, а обычные тексты — отрицательным) для IsolationForest. В качестве признаков используйте TF-IDF, где словарь и IDF строятся по обучающей выборке. Не забудьте подобрать гиперпараметры.

In [4]:
print(x_test)

0        <p>my model:</p>\n\n<pre><code>class MyModel(m...
1        <pre><code>CREATE OR REPLACE PROCEDURE disable...
2        <p>I have an API call that returns an array of...
3        <p>I am trying to make a an area chart that ha...
4        <p><strong>Image:</strong>(<a href="https://i....
                               ...                        
10095    <p>I have a Dockerfile that is supposed to bui...
10096    <p>I am trying to use this code to resize my t...
10097    <p>Cannot convert lambda expression to type 'b...
10098    <p>I have <code>A.js</code>, <code>B.js</code>...
10099    <p>I am trying to plot figures in real time us...
Length: 10100, dtype: object


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score

vec = TfidfVectorizer(max_features=5000) 
x_train_v = vec.fit_transform(x_train)
x_test_v  = vec.transform(x_test)
iso = IsolationForest(
    n_estimators=100,
    max_samples='auto',
    contamination=0.01,
    random_state=42
)
iso.fit(x_train_v)
y_pred = iso.predict(x_test_v)




In [6]:
print(len(y_pred))
for i in range(len(x_test)):
    if(y_pred[i] == -1):

        print(x_test[i][:20])


10100
<p>We are porting ou
<p>I am getting the 
<p>I am working on a
<p>I read some threa
<p>I'm trying to get
<h2>Solved:  A frien
<p>I am trying to te
<p>I'm building a do
<p>I have a setTimeo
<p>I have an ASP.NET
<p>I have question, 
<p>We have a spring 
<p>I've created defa
From: JJMARVIN@pucc.
<p>I have a strange 
<p>On my site every 
<p>I'm writing a web
<p><strong>strong te
<p>I an using <a hre
From: dxf12@po.cwru.
<p>I am trying to ma
<p>Here's the proble
<p>I am making a Chr
<p>I am currently tr
<pre><code>def compr
<p>I'm in the proces
<p>I'm trying to pro
<p>When user clicks 
<p>I'm trying to cod
From: pharvey@quack.
<p>I am currently bu
<p>I am currently ru
<p>I'm working on a 
<p>I am using NSFetc
<p>I am working on a
<p>I'm new to using 
<p>My aim is to find
<p>I've been reading
<p>Suppose I have a 
From: jsledd@ssdc.sa
<p>I hope someone he
<p>So, I've been str
<p>I am drawing the 
<p>I'm using jQuery 
<p>So, I am currentl
<p>I've been reading
<p>So I have two tab
<p>I wa

In [7]:

y_pred = (y_pred == -1).astype(int)
print(y_pred)
pre = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
print(pre, rec)

[0 0 0 ... 0 0 0]
0.09482758620689655 0.11


**(5 баллов)**

Скорее всего, качество оказалось не на высоте. Разберитесь, в чём дело:
* посмотрите на тексты, которые выделяются как аномальные, а также на слова, соответствующие их ненулевым признакам
* изучите признаки аномальных текстов
* посмотрите на тексты из обучающей выборки, ближайшие к аномальным; действительно ли они похожи по признакам?

Сделайте выводы и придумайте, как избавиться от этих проблем. Предложите варианты двух типов: (1) в рамках этих же признаков (но которые, возможно, будут считаться по другим наборам данных) и методов и (2) без ограничений на изменения. Реализуйте эти варианты и проверьте их качество.

In [8]:
an_idx = np.where(y_pred == 1)[0]
print(len(y_pred))
for i in an_idx:
    
    print(x_test[i][0:300])
    pass


10100
<p>We are porting our C# application (.Net Core) to Fedora 29 on ARM64 machines.
I am tasked with building our developing environment.
Currently I have a remote desktop with VNC and the next thing I need is an IDE.
So far I tried installing:</p>

<ul>
<li>MonoDevelop - Opened a window, showed an err
<p>I am getting the <code>SQLGrammarException</code> exception while using the following code</p>
<p>Please check the my Repository class which are as follow</p>
<pre><code>package repository;

package repository;

import java.util.List;

import org.springframework.data.jpa.repository.JpaRepository;
<p>I am working on a large scale project where a custom (pretty good and robust) framework has been provided and we have to use that for showing up forms and views.</p>
<hr />
<p>There is abstract class StrategyEditor (derived from some class in framework) which is instantiated whenever a new Strate
<p>I read some threads about how the javascript function parameters passing works when the 

In [9]:
feature_names = np.array(vec.get_feature_names_out())

def top_words(row, topn=10):
    arr = row.toarray().ravel()
    idx = np.argsort(arr)[-topn:]
    return feature_names[idx]

for i in an_idx[:5]:
    print(top_words(x_test_v[i]))

['failed' 'seem' 'far' 'our' 'environment' 'match' 'linux' 'scripts'
 'install' 'li']
['invoke' 'quot' 'apache' 'catalina' 'hibernate' 'applicationfilterchain'
 'loader' 'springframework' 'org' 'java']
['is' 'that' 'opened' 'strong' 'code' 'we' 'memory' 'objects' 'handles'
 'br']
['copy' 'parameter' 'passed' 'some' 'properties' 'function' 'passing'
 'memory' 'the' 'object']
['happens' 'card' 'scene' 'pass' 'test' 'li' 'the' 'buffer' 'shader'
 'fragment']


In [10]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=3, metric='cosine')
nn.fit(x_train_v)

dist, ind = nn.kneighbors(x_test_v[an_idx])

for i, idx in enumerate(an_idx):
    print("anom:")
    print(x_test.iloc[idx][:20])
    print("\nknn:")
    for j in ind[i]:
        print("-", x_train.iloc[j][:20])
    print("="*80)


anom:
<p>We are porting ou

knn:
- <p>I’m thinking abou
- <p>I am trying to bu
- <p>Entity Framework 
anom:
<p>I am getting the 

knn:
- <p>I'm using the fol
- <p>In my Spring Boot
- <p>I recently upgrad
anom:
<p>I am working on a

knn:
- <p>file looks like t
- <p>I was told to cre
- <p>I've spent many h
anom:
<p>I read some threa

knn:
- <p>I need a class to
- <p>On collection, th
- <p>I have a number o
anom:
<p>I'm trying to get

knn:
- <p>I have a GLSL pro
- <p>I've written a sm
- <p>In my activity, I
anom:
<h2>Solved:  A frien

knn:
- <p>I have a .Net Cor
- <p>I'm trying to par
- <p>I have multiple c
anom:
<p>I am trying to te

knn:
- <p>I am using an onl
- <p>I have a question
- <p>i want to do this
anom:
<p>I'm building a do

knn:
- <p>I had letsencrypt
- <p>I have a location
- <p>I am getting <cod
anom:
<p>I have a setTimeo

knn:
- <p>I know there is a
- <p>I am trying to de
- <p>I am trying to co
anom:
<p>I have an ASP.NET

knn:
- <p>I am using an API
- <p>I struggle with a
- <

### Эксперимент только с изменением датасета

In [11]:
import re
import pandas as pd

def sep_html(text):
    text = re.sub(r'(<[^>]+>)', r' \1 ', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()
    
x_train = x_train.apply(sep_html)
x_test = x_test.apply(sep_html)


print(x_test)
x_test

0        <p> my model: </p> <pre> <code> class MyModel(...
1        <pre> <code> CREATE OR REPLACE PROCEDURE disab...
2        <p> I have an API call that returns an array o...
3        <p> I am trying to make a an area chart that h...
4        <p> <strong> Image: </strong> ( <a href="https...
                               ...                        
10095    <p> I have a Dockerfile that is supposed to bu...
10096    <p> I am trying to use this code to resize my ...
10097    <p> Cannot convert lambda expression to type '...
10098    <p> I have <code> A.js </code> , <code> B.js <...
10099    <p> I am trying to plot figures in real time u...
Length: 10100, dtype: object


0        <p> my model: </p> <pre> <code> class MyModel(...
1        <pre> <code> CREATE OR REPLACE PROCEDURE disab...
2        <p> I have an API call that returns an array o...
3        <p> I am trying to make a an area chart that h...
4        <p> <strong> Image: </strong> ( <a href="https...
                               ...                        
10095    <p> I have a Dockerfile that is supposed to bu...
10096    <p> I am trying to use this code to resize my ...
10097    <p> Cannot convert lambda expression to type '...
10098    <p> I have <code> A.js </code> , <code> B.js <...
10099    <p> I am trying to plot figures in real time u...
Length: 10100, dtype: object

In [ ]:
vec = TfidfVectorizer(
    max_features=5000
)
x_train_v = vec.fit_transform(x_train)
x_test_v = vec.transform(x_test)
iso = IsolationForest(
    n_estimators=100,
    max_samples="auto",
    contamination=0.01,
    random_state=42,
)
iso.fit(x_train_v)
y_pred = iso.predict(x_test_v)


y_pred = (y_pred == -1).astype(int)
print(y_pred)
pre = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
print(pre, rec)

[0 0 0 ... 0 0 0]
0.031007751937984496 0.04


### Эксперимент с любыми изменениями

In [15]:
import re
import html
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score
from sklearn.neighbors import NearestNeighbors


def getn(text):


    code_ptn = re.compile(r'(<pre.*?>.*?</pre>|<code.*?>.*?</code>|```[\s\S]*?```)', flags=re.I)
    html_tag_ptn = re.compile(r'(<[^>]+>)')
    email_header_ptn = re.compile(r'(?m)^(From|Subject|Reply-To|Organization|In article|Lines|Date):')

    quote_ptn = re.compile(r'(?m)^(>+)\s*')
    url_ptn = re.compile(r'https?://\S+|www\.\S+')
    email_addr_ptn = re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')

    s = str(text)
    s = html.unescape(s)

    codes = code_ptn.findall(s)
    n_code_blocks = len(codes)
    s = code_ptn.sub(' __CODE_BLOCK__ ', s)

    tags = html_tag_ptn.findall(s)
    n_html_tags = len(tags)
    s = html_tag_ptn.sub(' __HTML_TAG__ ', s)

    n_email_headers = sum(1 for _ in email_header_ptn.finditer(s)) 
    s = email_header_ptn.sub(' __EMAIL_HEADER__ ', s)

    n_urls = len(url_ptn.findall(s))
    s = url_ptn.sub(' __URL__ ', s)

    n_emails = len(email_addr_ptn.findall(s)) 
    s = email_addr_ptn.sub(' __EMAIL_ADDR__ ', s)


    s = re.sub(r'\s+', ' ', s).strip()

    ft = {
        # 'n_code_blocks': n_code_blocks,
        # 'n_html_tags': -n_html_tags,
        'n_email_headers': n_email_headers,
        # 'n_urls': n_urls,
        'n_emails': n_emails,
    }

    return s, ft


In [16]:
def bf(x_train, x_test):
    alls = pd.concat([x_train, x_test], ignore_index=True)

    clt = []
    ftrtr = []

    for t in alls:
        c, f = getn(t)
        clt.append(c)
        ftrtr.append(f)

    df_features = pd.DataFrame(ftrtr).fillna(0)

    n_train = len(x_train)

    clean_train = pd.Series(clt[:n_train])
    clean_test = pd.Series(clt[n_train:])

    df_train = df_features.iloc[:n_train].reset_index(drop=True)
    df_test = df_features.iloc[n_train:].reset_index(drop=True)

    return clean_train, clean_test, df_train, df_test


In [17]:

from sklearn.preprocessing import StandardScaler
from scipy import sparse

scaler = StandardScaler()


def bvct(clean_train, clean_test, df_train, df_test):
    vec = TfidfVectorizer(
        max_features=3000,
        stop_words='english',
        min_df=3,
        ngram_range=(1,2),
        sublinear_tf=True
    )

    X_tfidf_train = vec.fit_transform(clean_train)
    X_tfidf_test = vec.transform(clean_test)

    scaler = StandardScaler()
    X_num_train = scaler.fit_transform(df_train.values)
    X_num_test = scaler.transform(df_test.values)
    coef = 100000000
    X_num_train = scaler.fit_transform(df_train.values * coef)
    X_num_test  = scaler.transform(df_test.values * coef)
    X_train = sparse.hstack([X_tfidf_train, sparse.csr_matrix(X_num_train)])
    X_test = sparse.hstack([X_tfidf_test, sparse.csr_matrix(X_num_test)])
    
    X_train = sparse.hstack([sparse.csr_matrix(X_num_train)])
    X_test = sparse.hstack([sparse.csr_matrix(X_num_test)])

    return X_train, X_test, vec


In [18]:
def run(X_train, X_test, y_test):
    iso = IsolationForest(
        n_estimators=200,
        contamination=0.00990099009900990099009900990099,
        random_state=42
    )

    iso.fit(X_train)
    scores = iso.decision_function(X_test)  # array float
    # threshold_99 = np.percentile(scores, 100-0.990099009900990099009900990099)

    # y_pred = (scores >= threshold_99).astype(int)  # 1 = аномалия
    y_pred = (scores < 0).astype(int)  # 1 = аномалия

    


    pre = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    print("Precision:", pre)
    print("Recall:", rec)

    return y_pred


In [19]:
def showan(y_pred, clean_test, df_features_test, n=10):
    idx = np.where(y_pred == 1)[0]

    print("Tot:", len(idx))

    for i in idx[:n]:
        print("\n" + "-"*70)
        print("Text:")
        print(clean_test.iloc[i][:400])
        print("\nNumeric ft:")
        print(df_features_test.iloc[i].to_dict())


In [20]:
clean_train, clean_test, df_train, df_test = bf(x_train, x_test)

X_train, X_test, vec = bvct(
    clean_train, clean_test,
    df_train, df_test
)
# df_train['n_email_headers'] *= 10000000
# df_test['n_email_headers']  *= 10000000

y_pred = run(X_train, X_test, y_test)

showan(y_pred, clean_test, df_test, n=110)


Precision: 0.8448275862068966
Recall: 0.98
Tot: 116

----------------------------------------------------------------------
Text:
__EMAIL_HEADER__ __EMAIL_ADDR__ (Rick Granberry) Subject: Re: Help Reply-To: __EMAIL_ADDR__ (Rick Granberry) Organization: Motorola Paging and Telepoint Systems Group Lines: 46 In article __HTML_TAG__ , __EMAIL_ADDR__ (William Hargreaves) writes: > Hi everyone, > I'm a commited Christian that is battling with a problem. I > know that romans talks about how we are saved by our faith not our > deeds

Numeric ft:
{'n_email_headers': 1, 'n_emails': 3}

----------------------------------------------------------------------
Text:
__EMAIL_HEADER__ __EMAIL_ADDR__ (vera shanti noyes) Subject: Re: Easter: what's in a name? (was Re: New Testament Double Standard? Reply-To: __EMAIL_ADDR__ Organization: University of Chicago Lines: 26 In article __HTML_TAG__ __EMAIL_ADDR__ (Jayne Kulikauskas) writes: > __EMAIL_ADDR__ (Seanna (S.M.) Watson) writes: > >> In Quebec French, 

Подготовьте выборку: удалите столбцы `['id', 'date', 'price', 'zipcode']`, сформируйте обучающую и тестовую выборки по 10 тысяч домов.

Добавьте в тестовую выборку 10 новых объектов, в каждом из которых испорчен ровно один признак — например, это может быть дом из другого полушария, из далёкого прошлого или будущего, с площадью в целый штат или с таким числом этажей, что самолётам неплохо бы его облетать стороной.

Посмотрим на методы обнаружения аномалий на более простых данных — уж на табличном датасете с 19 признаками всё должно работать как надо!

Скачайте данные о стоимости домов: https://www.kaggle.com/harlfoxem/housesalesprediction/data

In [21]:
#code here

**Задание 9. (2 балла)**

Примените IsolationForest для поиска аномалий в этих данных, запишите их качество (как и раньше, это pre и rec). Проведите исследование:

Нарисуйте распределения всех признаков и обозначьте на этих распределениях объекты, которые признаны аномальными.

In [22]:
#code here